# Pinecone Tutorial for Your Medical RAG Chatbot

This notebook teaches Pinecone using the same design as your `knowledge.py` and `ingest.py` files.

The central pipeline is:

```text
PDF → text chunks → embeddings → Pinecone
                           ↑
User question → embedding ┘
                 ↓
          cosine similarity
                 ↓
          top-k chunks + metadata
                 ↓
              LangGraph
                 ↓
             final answer
```

We will first use tiny examples, then reproduce the logic of your chatbot.

## 1. What Pinecone stores

Pinecone stores **vectors** plus IDs and metadata. A record looks conceptually like:

```python
{
    "id": "abc123...",
    "values": [0.013, -0.287, ..., 0.042],
    "metadata": {
        "page": 42,
        "text": "Diabetes mellitus is ...",
        "file": "medical_book.pdf"
    }
}
```

The vector is used for similarity search. Metadata lets your chatbot recover the original chunk, page number, and filename.

## 2. Imports and configuration

This notebook can use your existing `app.config.settings`, or environment variables if run separately.

In [1]:
import os
import hashlib
from langchain_openai import ChatOpenAI
from pinecone import Pinecone, ServerlessSpec

try:
    from app.config import settings
    OPENAI_API_KEY = settings.openai_api_key
    PINECONE_API_KEY = settings.pinecone_api_key
    PINECONE_INDEX = settings.pinecone_index
    EMBED_MODEL = settings.openai_embed_model
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    BASE_URL = os.getenv("BASE_URL")
    PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
    PINECONE_INDEX = os.getenv("PINECONE_INDEX", "medical-rag-demo")
    EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

BASE_URL = os.getenv("BASE_URL")

print("Index:", PINECONE_INDEX)
print("Embedding model:", EMBED_MODEL)
print("OpenAI configured:", bool(OPENAI_API_KEY))
print("Pinecone configured:", bool(PINECONE_API_KEY))

Index: medical-agent-demo
Embedding model: text-embedding-3-small
OpenAI configured: True
Pinecone configured: True


## 3. Create the clients

Your `knowledge.py` caches the OpenAI and Pinecone clients with `@lru_cache()`. Here we create them directly so the notebook stays transparent.

In [2]:
if not OPENAI_API_KEY or not PINECONE_API_KEY:
    raise ValueError("Set OPENAI_API_KEY and PINECONE_API_KEY first.")

ai = ChatOpenAI(api_key=OPENAI_API_KEY,
                base_url=BASE_URL,
                model="gpt-5.6-luna",
                temperature=0.7,
                )

pc = Pinecone(api_key=PINECONE_API_KEY)

## 4. Create or connect to the index

Your `ensure_index()` creates a serverless Pinecone index using:

- dimension = `1536`
- metric = `cosine`
- AWS `us-east-1`

The index dimension **must match the embedding size**.

Cosine similarity is:

$$\cos(\theta)=\frac{A\cdot B}{\|A\|\|B\|}$$

Similar meanings should produce vectors pointing in similar directions.

In [10]:
PINECONE_API_KEY
pc

Pinecone(api_key='...rVPy', host='https://api.pinecone.io')

In [8]:
import os
from pinecone import Pinecone

# Test with the freshly generated key value
pc = Pinecone(api_key=PINECONE_API_KEY)

try:
    indexes = pc.list_indexes()
    print("✅ Successfully connected! Indexes:", [idx.name for idx in indexes])
except Exception as e:
    print("❌ Error:", e)

❌ Error: [403] <html><head>
<meta http-equiv="content-type" content="text/html;charset=utf-8">
<title>403 Forbidden</title>
</head>
<body text=#000000 bgcolor=#ffffff>
<h1>Error: Forbidden</h1>
<h2>Your client does not have permission to get URL <code>/indexes</code> from this server.</h2>
<h2></h2>
</body></html>


In [12]:
pc = Pinecone(api_key=PINECONE_API_KEY)

try:
    print(pc.list_indexes())
except Exception as e:
    print(type(e).__name__)
    print(e)

ForbiddenError
[403] <html><head>
<meta http-equiv="content-type" content="text/html;charset=utf-8">
<title>403 Forbidden</title>
</head>
<body text=#000000 bgcolor=#ffffff>
<h1>Error: Forbidden</h1>
<h2>Your client does not have permission to get URL <code>/indexes</code> from this server.</h2>
<h2></h2>
</body></html>


## 5. Embeddings

This mirrors your `embed()` function. OpenAI turns each text into one fixed-length numeric vector.

In [ ]:
def embed(texts):
    result = ai.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in result.data]

vecs = embed([
    "Diabetes can cause increased thirst.",
    "The heart pumps blood through the body."
])

print("Number of vectors:", len(vecs))
print("Vector dimension:", len(vecs[0]))
print("First 8 values:", vecs[0][:8])

## 6. Tiny Pinecone example

Before indexing a giant medical PDF, use three sentences. This is much easier to reason about than staring at thousands of vectors and pretending that is educational.

In [ ]:
demo_chunks = [
    {"id":"demo-diabetes", "page":10,
     "text":"Common symptoms of diabetes include increased thirst, frequent urination, and fatigue."},
    {"id":"demo-hypertension", "page":20,
     "text":"Hypertension means persistently elevated blood pressure and may have no obvious symptoms."},
    {"id":"demo-asthma", "page":30,
     "text":"Asthma can cause wheezing, coughing, chest tightness, and shortness of breath."},
]

vectors = embed([x["text"] for x in demo_chunks])
records = [
    {
        "id": item["id"],
        "values": vector,
        "metadata": {"page": item["page"], "text": item["text"], "file":"teaching-demo"}
    }
    for item, vector in zip(demo_chunks, vectors)
]

index.upsert(vectors=records)
print("Inserted", len(records), "vectors")

## 7. What `upsert` means

`upsert` means:

- insert if the ID does not exist
- update/replace if the same ID already exists

This is why stable IDs are useful.

## 8. Query Pinecone

The question is embedded with the same embedding model, then Pinecone finds the nearest stored vectors.

In [ ]:
query = "Why am I thirsty and urinating often?"
query_vector = embed([query])[0]

result = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

for match in result.matches:
    print("score:", round(match.score, 3))
    print("page :", match.metadata.get("page"))
    print("text :", match.metadata.get("text"))
    print("-" * 80)

## 9. `top_k` versus score threshold

Your code uses `top_k=5`, meaning Pinecone returns up to the five nearest vectors.

Then your code applies an additional threshold:

```python
if m.score >= 0.30
```

So these are different ideas:

- `top_k`: how many nearest candidates to request
- threshold: how weak a match you are willing to keep

The `0.30` threshold is a project choice, not a universal Pinecone rule.

In [ ]:
for k in [1, 2, 3]:
    r = index.query(vector=query_vector, top_k=k, include_metadata=True)
    print(f"\ntop_k={k}")
    for m in r.matches:
        print(round(m.score, 3), "->", m.metadata.get("text", "")[:70])

## 10. Your chunking function

Your `ingest.py` uses:

- chunk size = `1100` characters
- overlap = `180` characters
- step = `920` characters

The overlap helps preserve sentences or facts that cross a chunk boundary.

In [ ]:
def chunks(text, size=1100, overlap=180):
    text = " ".join(text.split())
    step = size - overlap
    return [
        text[i:i + size]
        for i in range(0, len(text), step)
        if len(text[i:i + size]) > 80
    ]

sample = ("Diabetes mellitus is a metabolic disease. " * 100).strip()
parts = chunks(sample)

print("Number of chunks:", len(parts))
print("First chunk length:", len(parts[0]))
print("Second chunk length:", len(parts[1]))
print("\nEnd of chunk 1:\n", parts[0][-220:])
print("\nStart of chunk 2:\n", parts[1][:220])

## 11. Deterministic IDs

Your ingestion code hashes:

```text
filename + page number + chunk number + chunk content
```

with SHA-256. Unchanged content gets the same ID on re-ingestion.

In [ ]:
pdf_name = "medical_book.pdf"
page_number = 15
part = 2
content = "Example medical text chunk."

key = hashlib.sha256(
    f"{pdf_name}:{page_number}:{part}:{content}".encode()
).hexdigest()

print(key)
print("ID length:", len(key))

## 12. Why batch 50 chunks?

Your ingestion script accumulates 50 chunks, sends all 50 texts in one embedding request, then upserts the resulting vectors together.

This is more efficient than one API request per chunk.

## 13. Metadata

Your real ingestion stores:

```python
metadata = {
    "page": page_number,
    "text": content,
    "file": pdf.name,
}
```

The embedding is used to find a match. Metadata is what lets the chatbot display the matched text and page number afterwards.

## 14. Miniature version of your ingestion pipeline

In [ ]:
pages = {
    1: "Diabetes mellitus is characterized by abnormal glucose regulation. " * 25,
    2: "Asthma is associated with airway inflammation and variable airflow obstruction. " * 25,
}

batch = []
for page_number, page_text in pages.items():
    for part, content in enumerate(chunks(page_text), start=1):
        key = hashlib.sha256(
            f"teaching.pdf:{page_number}:{part}:{content}".encode()
        ).hexdigest()
        metadata = {"page": page_number, "text": content, "file":"teaching.pdf"}
        batch.append((key, content, metadata))

vectors = embed([b[1] for b in batch])
index.upsert(vectors=[
    {"id": b[0], "values": v, "metadata": b[2]}
    for b, v in zip(batch, vectors)
])

print("Inserted", len(batch), "chunks")

## 15. Reproduce your `search_kb()` function

In [ ]:
def search_kb(query, top_k=5, threshold=0.30):
    vector = embed([query])[0]
    result = index.query(vector=vector, top_k=top_k, include_metadata=True)
    return [
        {
            "page": m.metadata.get("page"),
            "text": m.metadata.get("text", ""),
            "score": round(m.score, 3),
            "source": m.metadata.get("file", "unknown")
        }
        for m in result.matches
        if m.score >= threshold
    ]

search_kb("What symptoms are associated with diabetes?")

## 16. How Pinecone connects to LangGraph

Pinecone performs retrieval, not answer generation.

```text
question
   ↓
search_kb(question)
   ↓
question embedding
   ↓
Pinecone similarity search
   ↓
relevant chunks
   ↓
LangGraph state['kb']
   ↓
LLM receives evidence
   ↓
final grounded answer
```

So Pinecone is the external knowledge retrieval layer. The LLM still generates the final response.

## 17. Inspect index statistics

Useful when checking whether ingestion actually put anything into Pinecone.

In [ ]:
index.describe_index_stats()

## 18. One important issue in your current ingestion comment

Your script says changed content produces a new ID and old chunks get replaced.

The first part is true. Changed content produces a new hash.

But a **new ID does not automatically delete the old ID** from Pinecone. The old vector can remain unless you explicitly delete stale records, clear/rebuild the index, or use a versioning/namespace strategy.

That matters when you repeatedly ingest edited PDFs.

## 19. Exercises

1. Change the query and inspect the ranking.
2. Compare `top_k=2`, `5`, and `10`.
3. Compare thresholds `0.20`, `0.30`, `0.50`, and `0.70`.
4. Add a metadata field such as `chapter`.
5. Change chunk size and overlap, then inspect retrieval quality.
6. Use 10 real medical questions and manually judge whether the retrieved top-5 chunks are relevant.
7. Try a question unrelated to the book and inspect the scores.

## 20. Final mental model

| Component | Job |
|---|---|
| PDF loader | Extract text |
| Chunker | Split text into retrieval units |
| OpenAI embeddings | Convert text into vectors |
| Pinecone | Store and search vectors |
| Metadata | Preserve page/text/file information |
| LangGraph | Decide how retrieval is used |
| LLM | Generate the final answer |

The key idea is:

**Pinecone searches semantic similarity between embeddings. It does not itself generate the medical answer.**